# Week 3: Word Meaning

**DATA/MSML 641: Natural Language Processing**  
**Instructor**: Armin Mehrabian  
**Date**: September 15-16, 2025

---

## 🎯 Learning Objectives

By the end of this lecture, you will be able to:
1. Distinguish between different approaches to representing word meaning
2. Understand lexicographic traditions and word sense distinctions
3. Apply decompositional and ontological approaches to semantic analysis
4. Implement word sense disambiguation techniques
5. Create and analyze distributional word representations


## 📋 Before we begin

**Assignment 1 is due today!**

**Last lecture recap**: 
- What constitutes a word (hidden complexity!)
- Multi-word expressions (MWE) and collocations
- Statistical hypothesis testing

**Today's focus**:
- **Word** - We've seen the complexity of defining words
- **Meaning** - What do we mean by meaning? 🤔

## 🌍 Why Word Meaning Matters: Real-World Impact

Before diving into technical approaches, let's see how word meaning representations power technologies you use every day:

### **🔍 Search & Discovery**
- **Google Search**: Uses semantic understanding to match "car repairs" with "automotive maintenance"
- **Amazon Product Search**: Finds "wireless headphones" when you search for "bluetooth earbuds"
- **Netflix Recommendations**: Understands that "sci-fi thriller" relates to specific genre preferences

### **🗣️ Language Translation**
- **DeepL vs Google Translate (2024)**: Different approaches to handling ambiguous words
  - Swedish "kön" → "queue" or "gender"? Context determines meaning
  - Technical terms require specialized sense disambiguation

### **🤖 Virtual Assistants**
- **Siri/Alexa**: "Play some jazz" requires understanding music genres
- **Customer Support Bots**: Distinguish between "account balance" (financial) vs "work-life balance" (wellness)

### **📊 Business Intelligence**
- **Social Media Analytics**: Sentiment analysis of product mentions
- **Financial News**: Extracting market-moving information from text
- **Medical Records**: Identifying symptoms and treatments from clinical notes

**Today's Question**: How do we teach machines to understand word meanings as flexibly as humans do?

# 1. Lexicographic Approaches

## The Traditional Dictionary-Making Approach

### The Lexicographic Tradition

- **Lexicography**: The art and science of constructing dictionaries
- **Purpose**: Enumerate word meanings (senses)
- **Key Question**: What are definitions?

**Important insight**: 
> Definitions are not "defining" in the mathematical or legal sense. They are **text you read that evoke concepts in your head**.

**Example**: Legal definition of "assault"
- Intentional, affects another person with reasonable apprehension  
- Harmful or offensive contact, imminently
- Physical injury **NOT** required

### 🏢 **Modern Applications**

**Google's Knowledge Graph** uses lexicographic principles:
- Entities have multiple senses: "Apple" (company) vs "Apple" (fruit)  
- Each sense has structured properties and relationships
- Powers search understanding and featured snippets

**Legal Tech Companies** (e.g., Westlaw, LexisNexis):
- Maintain precise definitions for legal terms
- Context determines which legal sense applies
- Critical for case law research and compliance

### Forms of Word Sense Distinction

#### 1. **Homonymy** 
Different words that happen to have the same spelling/pronunciation

**Example: "Pen"**
- "Place to keep animals" (enclosure)
- "Instrument to write with" (writing tool)
- *Question*: Coincidental or historical relationship?

#### 2. **Polysemy**
Single word with multiple **related** meanings

**Example: "Bass"**
- "Low register" (music)
- "Person who sings low" (singer)
- *Note*: Related meanings, not coincidental

#### 3. **Systematic/Structured Polysemy**
Polysemy that applies systematically to a category

**Example**: Animal ↔ Food alternation
- chicken, lamb, goose, turkey, duck...
- Each has both an **animal sense** and a **food sense**

In [ ]:
# Interactive Exercise: Exploring Word Senses
import pandas as pd

# Examples of different types of ambiguity
word_examples = {
    'Word': ['bank', 'bark', 'chicken', 'bass', 'mouse', 'turkey'],
    'Sense 1': ['financial institution', 'dog sound', 'bird', 'low voice/music', 'small animal', 'bird'],
    'Sense 2': ['river edge', 'tree covering', 'meat', 'fish', 'computer device', 'silly person'],
    'Type': ['homonymy', 'homonymy', 'systematic polysemy', 'polysemy', 'polysemy', 'polysemy']
}

df = pd.DataFrame(word_examples)
print("🔍 Word Sense Examples")
print(df.to_string(index=False))

print("\n💭 Discussion Questions:")
print("1. Can you identify which examples show homonymy vs. polysemy?")
print("2. What other animal/food pairs can you think of?")
print("3. How would you determine if two senses are related or coincidental?")

### Dictionary-Making Approaches

#### **Labor-Intensive Tradition**
- **Oxford English Dictionary (OED)**: Started 1857
- **Crowdsourcing**: Volunteers found quotations for each word
- **Timeline**: More than 50 years to complete!
- **Reference**: Simon Winchester, *The Professor and the Madman*

#### **Corpus Analysis Approach**
- **Collins COBUILD Dictionary**: Published 1987
- **Method**: KWIC (Keyword in Context) analysis
- **Data-driven**: Based on actual language usage patterns

In [ ]:
# Demo: KWIC (Keyword in Context) Analysis
import re

def kwic_analysis(text, target_word, window=3):
    """Extract KWIC concordances for a target word"""
    words = text.lower().split()
    concordances = []
    
    for i, word in enumerate(words):
        if target_word.lower() in word:
            # Extract context window
            start = max(0, i - window)
            end = min(len(words), i + window + 1)
            
            left_context = " ".join(words[start:i])
            right_context = " ".join(words[i+1:end])
            
            concordances.append({
                'left': left_context,
                'keyword': word,
                'right': right_context
            })
    
    return concordances

# Example sentences with "bass"
bass_examples = """
He will sing a bass solo after the intermission.
She has a lovely bass voice that resonates well.
The pianist struggled to reach the bass notes on the lower octave.
The band played bass guitar and drums loudly.
The ensemble featured a bass and a lead guitarist performing together.
We caught several bass while fishing in the lake.
The striped bass is common in these coastal waters.
"""

# Analyze "bass" in context
concordances = kwic_analysis(bass_examples, "bass")

print("🎵 KWIC Analysis: 'bass'")
print("=" * 60)
for i, conc in enumerate(concordances, 1):
    print(f"{i:2d}. {conc['left']:25} {conc['keyword']:8} {conc['right']}")

print("\n💡 Observations:")
print("- Can you identify which uses refer to music vs. fish?")
print("- What contextual clues help disambiguation?")

# 2. Decompositional Approaches

## Breaking Down Meaning into Components

### Lexical Decomposition Theory

**Core assumption**: Lexical meaning is complex and can be expressed in structured meaning representations

**Example sentence**: "Jill likes Jack most of the time"
- **Predicates**: likes
- **Arguments**: Jill & Jack  
- **Operators**: "what", "most of the time"

### Approach 1: Necessary and Sufficient Conditions

**Traditional approach** (Katz and Fodor): Definitions provide necessary and sufficient conditions

**Example**: "bachelor" iff {human, male, adult, never married}

#### **Challenge**: Very hard to identify necessary and sufficient conditions!

**Question**: Is having four legs a necessary condition for being a dog?  
→ Does a dog that lost a leg in an accident no longer qualify as a dog? 🤔

In [ ]:
# Interactive Exercise: Necessary vs. Sufficient Conditions

def analyze_concept_conditions(concept, proposed_features):
    """Analyze whether features are necessary/sufficient for a concept"""
    print(f"🔍 Analyzing concept: '{concept}'")
    print(f"Proposed features: {proposed_features}")
    
    print("\n🤔 Consider these edge cases:")
    return proposed_features

# Test traditional definitions
concepts = {
    'dog': ['animal', 'four legs', 'barks', 'domesticated'],
    'bird': ['animal', 'has wings', 'can fly', 'has feathers'],
    'chair': ['furniture', 'has legs', 'for sitting', 'has back']
}

for concept, features in concepts.items():
    analyze_concept_conditions(concept, features)
    print("\n" + "="*50 + "\n")

print("💭 Discussion:")
print("- What about a three-legged dog? Still a dog?")
print("- What about penguins? Birds that can't fly?")
print("- What about a stool? Chair without a back?")
print("- How do fuzzy boundaries affect computational systems?")

### Approach 2: Structured Conceptual Representations

**Jackendoff's Conceptual Structure Theory**:

> "Conceptual structure is not a part of language per se – it is a part of **thought**. It is the locus for understanding linguistic utterances in context, incorporating pragmatic considerations and 'world knowledge'."

### Lexical-Conceptual Structure (LCS)

**Semantic primitives**: CAUSE, GO, TO, FROM, etc.

**Example: "give"**
```
give: [CAUSE (x, [GO (y, [TO (z)])])]
```

**"John gave me the book"** →  
John (=X) CAUSED the book (=Y) to GO TO me (=Z)

In [ ]:
# Demo: Lexical Conceptual Structure Analysis

class LCSAnalyzer:
    """Simple LCS representation for verbs"""
    
    def __init__(self):
        self.verb_templates = {
            'give': 'CAUSE(X, GO(Y, TO(Z)))',
            'take': 'CAUSE(X, GO(Y, FROM(Z), TO(X)))',
            'put': 'CAUSE(X, GO(Y, TO(Z)))',
            'remove': 'CAUSE(X, GO(Y, FROM(Z)))',
            'send': 'CAUSE(X, GO(Y, TO(Z)))',
            'bring': 'CAUSE(X, GO(Y, TO(HERE)))'
        }
    
    def analyze_sentence(self, sentence, verb, arguments):
        """Analyze a sentence using LCS representation"""
        if verb.lower() in self.verb_templates:
            template = self.verb_templates[verb.lower()]
            print(f"📝 Sentence: {sentence}")
            print(f"🔧 Verb: {verb}")
            print(f"📊 LCS Template: {template}")
            
            if len(arguments) >= 3:
                x, y, z = arguments[0], arguments[1], arguments[2]
                print(f"\n🎯 Instantiated:")
                instantiated = template.replace('X', x).replace('Y', y).replace('Z', z)
                print(f"   {instantiated}")
                
                print(f"\n🔄 Interpretation:")
                print(f"   {x} CAUSED {y} to GO TO {z}")
            
        else:
            print(f"❌ No LCS template available for '{verb}'")
    
    def compare_verbs(self, verbs):
        """Compare LCS structures of multiple verbs"""
        print("🔄 LCS Comparison:")
        for verb in verbs:
            if verb in self.verb_templates:
                print(f"  {verb:8} → {self.verb_templates[verb]}")

# Create analyzer and test
lcs = LCSAnalyzer()

# Analyze example sentences
examples = [
    ("John gave Mary the book", "give", ["John", "the book", "Mary"]),
    ("Sarah took the keys from Tom", "take", ["Sarah", "the keys", "Tom"]),
    ("The robot put the box on the shelf", "put", ["The robot", "the box", "the shelf"])
]

for sentence, verb, args in examples:
    lcs.analyze_sentence(sentence, verb, args)
    print("\n" + "="*60 + "\n")

# Compare related verbs
lcs.compare_verbs(['give', 'take', 'put', 'send', 'bring'])

# 3. Ontological Approaches

## Organizing Knowledge in Hierarchies

### Ontologies

**Definition**: Sets of entities and relations between them

**Key relationship**: **IS-A** (subsumption) for concepts

**Examples**:
- Biological taxonomies (most familiar)
- Gene ontologies
- Business ontologies  
- Astronomy ontologies

### IS-A Inheritance

**Core principle**: C2 IS-A C1 ⟹ for all properties f of C1, f is also a property of C2

**Connection**: Related to subclassing in object-oriented programming!

In [ ]:
# Demo: Building a Simple Ontology
import networkx as nx
import matplotlib.pyplot as plt

class SimpleOntology:
    """A simple ontology with IS-A relationships"""
    
    def __init__(self):
        self.graph = nx.DiGraph()
        self.properties = {}  # concept -> set of properties
    
    def add_concept(self, concept, properties=None):
        """Add a concept with its properties"""
        self.graph.add_node(concept)
        self.properties[concept] = set(properties or [])
    
    def add_isa_relation(self, child, parent):
        """Add IS-A relation: child IS-A parent"""
        self.graph.add_edge(child, parent, relation="IS-A")
    
    def get_inherited_properties(self, concept):
        """Get all properties of a concept (including inherited ones)"""
        all_properties = set(self.properties.get(concept, []))
        
        # Inherit from parents
        for parent in self.graph.successors(concept):
            parent_props = self.get_inherited_properties(parent)
            all_properties.update(parent_props)
        
        return all_properties
    
    def display_hierarchy(self, root_concept):
        """Display the concept hierarchy"""
        def print_tree(concept, level=0):
            indent = "  " * level
            properties = self.properties.get(concept, set())
            props_str = f" [{', '.join(properties)}]" if properties else ""
            print(f"{indent}{concept}{props_str}")
            
            # Find children (concepts that have this as parent)
            children = [node for node in self.graph.nodes() 
                       if concept in self.graph.successors(node)]
            
            for child in children:
                print_tree(child, level + 1)
        
        print_tree(root_concept)

# Build animal ontology
ontology = SimpleOntology()

# Add concepts with properties
ontology.add_concept("Animal", ["alive", "moves", "reproduces"])
ontology.add_concept("Mammal", ["warm-blooded", "has_fur"])
ontology.add_concept("Fish", ["cold-blooded", "has_scales", "lives_in_water"])
ontology.add_concept("Dog", ["domesticated", "barks"])
ontology.add_concept("Bear", ["large", "omnivore"])
ontology.add_concept("Poodle", ["curly_fur", "intelligent"])
ontology.add_concept("Beagle", ["good_nose", "hunting_dog"])
ontology.add_concept("Remy", ["pet", "brown_fur"])  # instance

# Add IS-A relations
ontology.add_isa_relation("Mammal", "Animal")
ontology.add_isa_relation("Fish", "Animal")
ontology.add_isa_relation("Dog", "Mammal")
ontology.add_isa_relation("Bear", "Mammal")
ontology.add_isa_relation("Poodle", "Dog")
ontology.add_isa_relation("Beagle", "Dog")
ontology.add_isa_relation("Remy", "Beagle")  # INSTANCE-OF

print("🌳 Animal Ontology Hierarchy:")
print("=" * 40)
ontology.display_hierarchy("Animal")

print("\n🔍 Property Inheritance Examples:")
print("=" * 40)
test_concepts = ["Remy", "Poodle", "Dog", "Mammal"]
for concept in test_concepts:
    props = ontology.get_inherited_properties(concept)
    print(f"{concept:8}: {sorted(props)}")

### WordNet: The Most Famous Ontology

**Created by**: George Miller (mid-1980s, passed away 2012 at age 92)  
**Scope**: Originally English, now multilingual WordNets  
**Organization**: Separate ontologies by part of speech (N, V, Adj, Adv)

#### **Core Concept: Synset** (Synonym Set) ≈ "Concept"
- `⟨board, plank, beam⟩` - wood pieces
- `⟨board, committee, panel, council⟩` - governing bodies

#### **Key Relations**:
- **Hypernym/Hyponym** (IS-A): dog IS-A animal
- **Instance** (INSTANCE-OF): Remy INSTANCE-OF beagle  
- **Meronym** (PART-OF): wheel PART-OF car
- **Antonym** (OPPOSITE): hot OPPOSITE cold

In [ ]:
# Demo: Working with WordNet
import nltk
from nltk.corpus import wordnet as wn

# Download WordNet if not already available
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

def explore_wordnet_synsets(word):
    """Explore WordNet synsets for a word"""
    synsets = wn.synsets(word)
    
    print(f"🔍 WordNet Analysis: '{word}'")
    print(f"Found {len(synsets)} synset(s):\n")
    
    for i, synset in enumerate(synsets, 1):
        print(f"{i}. {synset.name()}")
        print(f"   Definition: {synset.definition()}")
        print(f"   Examples: {synset.examples()}")
        print(f"   Lemmas: {[lemma.name() for lemma in synset.lemmas()]}")
        
        # Show hypernyms (IS-A parents)
        hypernyms = synset.hypernyms()
        if hypernyms:
            print(f"   Hypernyms: {[h.name() for h in hypernyms]}")
        
        print()

def compare_word_similarity(word1, word2):
    """Compare semantic similarity using WordNet"""
    synsets1 = wn.synsets(word1)
    synsets2 = wn.synsets(word2)
    
    if not synsets1 or not synsets2:
        print(f"❌ No synsets found for {word1} or {word2}")
        return
    
    # Get most similar pair
    max_similarity = 0
    best_pair = None
    
    for s1 in synsets1:
        for s2 in synsets2:
            similarity = s1.path_similarity(s2)
            if similarity and similarity > max_similarity:
                max_similarity = similarity
                best_pair = (s1, s2)
    
    print(f"🔗 Similarity between '{word1}' and '{word2}': {max_similarity:.3f}")
    if best_pair:
        print(f"   Best match: {best_pair[0].name()} ↔ {best_pair[1].name()}")

# Explore examples
words_to_explore = ['dog', 'bass']
for word in words_to_explore:
    explore_wordnet_synsets(word)
    print("=" * 60)

# Compare word similarities
print("\n🔍 Semantic Similarity Examples:")
word_pairs = [('dog', 'cat'), ('dog', 'animal'), ('car', 'vehicle'), ('bass', 'guitar')]
for w1, w2 in word_pairs:
    compare_word_similarity(w1, w2)

# 4. Word Sense Disambiguation (WSD)

## A Classic Problem in NLP

### WSD as Classification Problem

#### **Problem Formulation**:
- **Given**: 
  - Enumerated senses {s₁, s₂, …, sₙ} for word w
  - Context for w: (…. w ….)
- **Select**: "Correct" sᵢ for w

#### **Simple Approaches**:
- **Strong baseline**: Most frequent sense (surprisingly effective!)
- **Context-based**: Look at surrounding words
- **Statistical methods**: We'll learn these in **Week 8: Machine Learning in NLP**

#### **Key Insight**: Context is crucial for disambiguation
- "The **bank** charged a fee" (financial institution)
- "We sat by the river **bank**" (edge/shore)

The surrounding words provide the clues needed to distinguish senses.

# Simple WSD Example: Context Clues
import pandas as pd

def analyze_word_context(sentences, target_word):
    """Simple analysis of context clues for word disambiguation"""
    
    print(f"🔍 Context Analysis for: '{target_word}'")
    print("=" * 50)
    
    examples = []
    
    for i, sentence in enumerate(sentences, 1):
        # Find the target word and extract context
        words = sentence.lower().split()
        
        for j, word in enumerate(words):
            if target_word.lower() in word:
                # Get surrounding words
                left_context = words[max(0, j-2):j]
                right_context = words[j+1:min(len(words), j+3)]
                
                examples.append({
                    'Sentence': sentence,
                    'Left Context': ' '.join(left_context),
                    'Target': word,
                    'Right Context': ' '.join(right_context),
                    'Likely Sense': ''  # Students can fill this in
                })
    
    # Display as table
    df = pd.DataFrame(examples)
    print(df.to_string(index=False))
    
    return examples

# Example sentences with "bass"
bass_sentences = [
    "She plays bass guitar in the jazz band.",
    "Her bass voice resonated through the auditorium.",
    "We caught several bass while fishing at the lake.",
    "The bass notes were difficult to reach on the piano.",
    "Striped bass are common in these coastal waters.",
    "The choir needs more singers in the bass section."
]

# Analyze context clues
examples = analyze_word_context(bass_sentences, "bass")

print("\n🤔 Discussion Questions:")
print("1. What context clues help you identify each sense?")
print("2. Which examples are easiest/hardest to disambiguate?") 
print("3. How would you teach a computer to use these clues?")
print("4. What happens when context clues are ambiguous or missing?")

In [ ]:
# Demo: Simple Word Sense Disambiguation
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
import re

class SimpleWSD:
    """Simple Word Sense Disambiguation using context features"""
    
    def __init__(self, target_word, window_size=2):
        self.target_word = target_word.lower()
        self.window_size = window_size
        self.vectorizer = CountVectorizer()
        self.classifier = MultinomialNB()
        self.sense_labels = []
    
    def extract_context_features(self, sentence):
        """Extract context window around target word"""
        words = sentence.lower().split()
        
        # Find target word positions
        target_positions = [i for i, word in enumerate(words) 
                          if self.target_word in word.lower()]
        
        contexts = []
        for pos in target_positions:
            # Extract window
            start = max(0, pos - self.window_size)
            end = min(len(words), pos + self.window_size + 1)
            
            # Remove target word and join context
            context_words = words[start:pos] + words[pos+1:end]
            contexts.append(" ".join(context_words))
        
        return contexts[0] if contexts else ""
    
    def train(self, training_data):
        """Train WSD classifier
        training_data: list of (sentence, sense_label) tuples
        """
        contexts = []
        labels = []
        
        for sentence, sense in training_data:
            context = self.extract_context_features(sentence)
            if context:  # Skip if target word not found
                contexts.append(context)
                labels.append(sense)
        
        # Vectorize contexts and train classifier
        X = self.vectorizer.fit_transform(contexts)
        self.classifier.fit(X, labels)
        self.sense_labels = list(set(labels))
        
        print(f"✅ Trained WSD model for '{self.target_word}'")
        print(f"   Senses: {self.sense_labels}")
        print(f"   Training examples: {len(contexts)}")
        print(f"   Feature vocabulary size: {len(self.vectorizer.vocabulary_)}")
    
    def predict(self, sentence):
        """Predict sense for target word in sentence"""
        context = self.extract_context_features(sentence)
        if not context:
            return None, 0.0
        
        X = self.vectorizer.transform([context])
        prediction = self.classifier.predict(X)[0]
        probabilities = self.classifier.predict_proba(X)[0]
        confidence = max(probabilities)
        
        return prediction, confidence
    
    def get_feature_importance(self, sense, top_n=5):
        """Get most important features for a sense"""
        if sense not in self.sense_labels:
            return []
        
        sense_idx = self.sense_labels.index(sense)
        feature_names = self.vectorizer.get_feature_names_out()
        feature_weights = self.classifier.feature_log_prob_[sense_idx]
        
        # Get top features
        top_indices = np.argsort(feature_weights)[-top_n:]
        top_features = [(feature_names[i], feature_weights[i]) for i in reversed(top_indices)]
        
        return top_features

# Create training data for "bass" disambiguation
bass_training_data = [
    # Musical sense
    ("He plays bass guitar in the band", "music"),
    ("Her bass voice resonates beautifully", "music"),
    ("The bass notes were too low to hear", "music"),
    ("She sings bass in the choir", "music"),
    ("The bass clef indicates low notes", "music"),
    ("His bass drum keeps the rhythm", "music"),
    ("The symphony has a prominent bass section", "music"),
    
    # Fish sense
    ("We caught three bass in the lake", "fish"),
    ("Striped bass are common in coastal waters", "fish"),
    ("The bass was swimming near the shore", "fish"),
    ("Large bass prefer deeper water", "fish"),
    ("Bass fishing is popular in spring", "fish"),
    ("The bass jumped out of the water", "fish"),
    ("We used worms to catch bass", "fish")
]

# Train WSD system
wsd = SimpleWSD("bass")
wsd.train(bass_training_data)

print("\n" + "=" * 50)

# Test on new sentences
test_sentences = [
    "The electric bass sounds great with distortion",
    "We saw a school of bass near the reef",
    "She has the lowest bass voice in the ensemble",
    "The angler caught a trophy bass",
    "The bass line drives the entire song"
]

print("🎯 WSD Predictions:")
for sentence in test_sentences:
    prediction, confidence = wsd.predict(sentence)
    print(f"   {sentence}")
    print(f"   → {prediction} (confidence: {confidence:.3f})\n")

# Show important features
print("🔍 Most Important Features:")
for sense in wsd.sense_labels:
    features = wsd.get_feature_importance(sense)
    print(f"\n{sense.upper()} sense:")
    for feature, weight in features:
        print(f"   {feature}: {weight:.3f}")

### "I don't believe in Word Senses" - Kilgariff (2003)

#### **Kilgariff's Arguments**:
- Advocated for **task-dependent clustering** of corpus instances
- **Schütze**: Early distributional representations can form natural clusters
- **Current trend**: Move away from discrete senses

#### **"You shall know a word by the company it keeps"**

This famous quote by J.R. Firth (1957) hints at an alternative approach to word meaning that we'll explore in detail in **Week 9: Vector Semantics & Embeddings**.

#### **However, consider challenges with discrete senses**:
- **Systematic regularities**: lamb as food vs. animal
- **Sparse data**: Especially in specialized domains
- **Explainability**: Supporting understanding and trusting inferences

### Why doesn't traditional WSD help many applications?

1. **Moderate performance** - Even good systems plateau around 80-85%
2. **Sense skew** - Most frequent sense dominates (Zipf's law)
3. **One sense per discourse** - Context usually disambiguates
4. **Implicit disambiguation** - Co-occurrence provides clues
   - Example: "bank" → ambiguous, but "bank ATM hours" → unambiguous
5. **Evaluation challenges** - Inter-annotator agreement often low

### **Real-World WSD Challenges**

**Translation Systems** still struggle with:
- **Context-dependent ambiguity**: Swedish "kön" → "queue" or "gender"?
- **Technical terminology**: Medical vs. general usage
- **Cultural references**: Idioms and colloquialisms

**This motivates alternative approaches to word meaning we'll cover in later weeks.**

In [ ]:
## 🔄 Summary & Key Takeaways

### **Three Approaches to Word Meaning (Week 3)**:

#### **1. Lexicographic** 📖
- **Method**: Traditional dictionary-making
- **Goal**: Enumerate discrete senses with definitions  
- **Strength**: Precise, interpretable
- **Challenge**: Labor-intensive, doesn't scale well
- **Example**: Oxford English Dictionary, WordNet synsets

#### **2. Decompositional** 🔧  
- **Method**: Break meaning into semantic primitives
- **Goal**: Compositional meaning representation
- **Strength**: Systematic, captures relationships
- **Challenge**: Hard to define primitives, coverage gaps
- **Example**: LCS representations like CAUSE(X, GO(Y, TO(Z)))

#### **3. Ontological** 🌳
- **Method**: Hierarchical organization of concepts
- **Goal**: Structured knowledge with inheritance
- **Strength**: Captures relationships, enables reasoning
- **Challenge**: Coverage, agreement on hierarchies
- **Example**: WordNet taxonomy, IS-A relationships

### **Word Sense Disambiguation**
- **Problem**: Given context, choose the right sense
- **Approaches**: Most frequent sense, context analysis
- **Challenge**: Low inter-annotator agreement
- **Reality**: Many applications work around WSD rather than solving it

### **Looking Ahead**
- **Week 4**: How do sequences of words create meaning?
- **Week 8**: Machine learning approaches to these problems  
- **Week 9**: Alternative approaches using distributional semantics

**Term Frequency - Inverse Document Frequency**

$$V_{w_i,d_j} = tf_{i,j} \times idf_i$$

Where:
- **$tf_{i,j}$**: Frequency of word $w_i$ in document $d_j$
- **$idf_i$**: Inverse document frequency 

$$idf_i = \log \frac{N}{\text{# docs containing } w_i}$$

- **N**: Total number of documents

**Usually modified to avoid zero division**:
$$idf_i = \log \frac{N}{1 + \text{# docs containing } w_i}$$

**Intuition**:
- Frequent words in a document → higher tf
- Rare words across corpus → higher idf  
- Common words like "the" get low scores (high denominator)

In [ ]:
# Demo: Building Distributional Word Vectors
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt

# Sample corpus
documents = [
    "The cat sat on the mat with the dog",
    "Dogs and cats are common pets in houses",
    "The kitten played with a ball of yarn",
    "Puppies and kittens are very cute animals",
    "The fish swims in the clear blue water",
    "Birds fly high in the bright blue sky",
    "The car drives fast on the highway",
    "Trucks and cars use gasoline for fuel",
    "The red apple tastes sweet and juicy",
    "Oranges and apples are healthy fruits",
    "The musician played bass guitar loudly",
    "We caught several bass fish in the lake",
    "Classical music often features violin and piano",
    "Fresh salmon and bass are popular fish"
]

# Create TF-IDF vectors
vectorizer = TfidfVectorizer(stop_words='english', max_features=50)
tfidf_matrix = vectorizer.fit_transform(documents)
feature_names = vectorizer.get_feature_names_out()

print(f"📊 TF-IDF Matrix: {tfidf_matrix.shape} (documents x features)")
print(f"Features: {feature_names[:10]}...")

# Build word co-occurrence matrix
def build_word_cooccurrence_matrix(documents, window_size=2):
    """Build word co-occurrence matrix from documents"""
    # Get vocabulary
    vocab = set()
    processed_docs = []
    
    for doc in documents:
        words = doc.lower().split()
        # Simple preprocessing
        words = [w.strip('.,!?"()') for w in words if len(w) > 2]
        processed_docs.append(words)
        vocab.update(words)
    
    vocab = sorted(list(vocab))
    vocab_to_idx = {word: idx for idx, word in enumerate(vocab)}
    
    # Initialize co-occurrence matrix
    matrix = np.zeros((len(vocab), len(vocab)))
    
    # Fill matrix
    for words in processed_docs:
        for i, word in enumerate(words):
            if word in vocab_to_idx:
                start = max(0, i - window_size)
                end = min(len(words), i + window_size + 1)
                
                for j in range(start, end):
                    if i != j and words[j] in vocab_to_idx:
                        matrix[vocab_to_idx[word]][vocab_to_idx[words[j]]] += 1
    
    return matrix, vocab

# Build co-occurrence matrix
cooc_matrix, vocab = build_word_cooccurrence_matrix(documents)
print(f"\n📈 Co-occurrence Matrix: {cooc_matrix.shape}")
print(f"Vocabulary: {vocab[:10]}...")

# Apply SVD for dimensionality reduction
svd = TruncatedSVD(n_components=10, random_state=42)
reduced_vectors = svd.fit_transform(cooc_matrix)

print(f"\n🔄 After SVD: {reduced_vectors.shape}")
print(f"Explained variance ratio: {svd.explained_variance_ratio_[:5]}")

# Function to find similar words
def find_similar_words(target_word, vectors, vocab, top_k=5):
    """Find most similar words using cosine similarity"""
    if target_word not in vocab:
        return f"Word '{target_word}' not in vocabulary"
    
    target_idx = vocab.index(target_word)
    target_vector = vectors[target_idx:target_idx+1]
    
    # Calculate similarities
    similarities = cosine_similarity(target_vector, vectors)[0]
    
    # Get top similar words (excluding the word itself)
    similar_indices = np.argsort(similarities)[::-1][1:top_k+1]
    
    results = []
    for idx in similar_indices:
        results.append((vocab[idx], similarities[idx]))
    
    return results

# Test word similarities
test_words = ['cat', 'dog', 'car', 'apple', 'bass']

print("\n🔍 Word Similarity Analysis:")
print("=" * 40)

for word in test_words:
    if word in vocab:
        similar = find_similar_words(word, reduced_vectors, vocab, top_k=3)
        print(f"\n'{word}' is similar to:")
        for similar_word, score in similar:
            print(f"  {similar_word}: {score:.3f}")
    else:
        print(f"\n'{word}' not found in vocabulary")

In [ ]:
## 📊 Basic Evaluation Concepts

Understanding how to evaluate word meaning systems helps us compare different approaches.

### **WSD Evaluation Basics**

#### **Accuracy** (Most Common)
```
Accuracy = Correct Predictions / Total Predictions
```
- Simple and interpretable
- "What percentage of word senses did we identify correctly?"

#### **Inter-Annotator Agreement**  
- **Problem**: Humans often disagree on word senses!
- **Example**: How many senses does "run" have? 
  - WordNet: 41 senses
  - OntoNotes: 12 senses
  - Most practical systems: 2-3 senses

#### **Most Frequent Sense Baseline**
- **Surprisingly strong**: Often 60-70% accuracy
- **Why**: Zipf's law - one sense usually dominates
- **Example**: "pen" usually means writing instrument, not animal enclosure

### **Evaluation Challenges**

#### **Sense Granularity Problem**
- **Fine-grained**: WordNet-level distinctions
- **Coarse-grained**: Practical application needs
- **Question**: Should we distinguish "bank" (financial) vs "bank" (investment)?

#### **Domain Dependence**
- "Cell" in biology vs telecommunications context
- Systems trained on news may fail on scientific text
- **Lesson**: Context and domain matter enormously

### **Alternative Success Measures**

Instead of linguistic accuracy, measure **practical utility**:
- **Search**: Do users find what they're looking for?
- **Translation**: Is the output understandable?
- **QA Systems**: Are the answers helpful?

**Key Insight**: Sometimes linguistically "imperfect" solutions work well in practice!

---

*Note: We'll learn more sophisticated evaluation methods in Week 8 (Machine Learning in NLP)*

## 💼 Real-World Applications of Week 3 Concepts

### **🎯 Discussion Questions for Aspiring NLP Engineers**

**How would you approach these using this week's methods?**

1. **Dictionary App Developer**: You're building a vocabulary app. Users look up "bank" and need to see different senses clearly organized. How would you structure this using lexicographic principles?

2. **Legal Tech Startup**: Your system processes contracts and needs to understand when "party" means "legal entity" vs "celebration". Which Week 3 approach is most relevant?

3. **Medical Information System**: Doctors write "patient presents chest pain" vs "patient has chest discomfort" - same concept, different words. How would ontological approaches help?

4. **Game Developer**: You're creating an AI dungeon master that needs to understand when players say "I attack the goblin with my staff" (weapon) vs "The wizard staff casts spells" (magical item). What's the core challenge?

### **🏢 Industry Examples Using These Approaches**

#### **Lexicographic Applications**
- **Dictionary.com, Merriam-Webster**: Still organize senses lexicographically
- **Siri/Alexa**: Use structured sense inventories for entity recognition
- **Legal databases (Westlaw)**: Maintain precise definitions for legal terms

#### **Ontological Applications**  
- **Google Knowledge Graph**: Uses IS-A relationships extensively
  - "Apple Inc." IS-A "Technology Company" IS-A "Organization"
- **Medical systems**: ICD-10 codes use hierarchical disease classification
- **E-commerce**: Product categorization (iPhone IS-A smartphone IS-A electronics)

#### **Decompositional Applications**
- **Question-answering**: "Who gave the book to Mary?" → CAUSE(X, GO(book, TO(Mary)))
- **Semantic parsing**: Converting natural language to database queries
- **Machine translation**: Understanding argument structure across languages

### **🤔 Limitations Students Should Understand**

1. **Scale Problem**: Hand-crafted approaches don't scale to web-sized data
2. **Coverage Gap**: No system covers all possible word meanings
3. **Agreement Problem**: Even experts disagree on sense boundaries  
4. **Context Dependence**: Same word can mean different things in different domains

**These limitations motivate the statistical and neural approaches we'll learn in Weeks 8-12!**

---

*Note: This gives you a foundation for understanding why the field moved toward data-driven methods.*

## 🤔 Discussion Questions

1. **Trade-offs**: What are the advantages and disadvantages of each approach to word meaning?

2. **Modern NLP**: Why have distributional approaches become dominant in modern NLP systems?

3. **Interpretability**: How do we balance performance with interpretability in meaning representations?

4. **Cultural aspects**: How do different approaches handle cultural and contextual variations in meaning?

5. **Future directions**: What might be the next breakthrough in computational semantics?

---

## 📚 Next Week Preview

**Topic**: Sequential Structure in Language  
**Focus**: Hidden Markov Models, sequence labeling, and part-of-speech tagging  
**Reading**: SLP Chapters 3, 8 (through 8.4), 9, Appendices A & B

### Prepare for next week:
- Review probability and Bayes' rule
- Think about how words depend on surrounding context
- Consider: How do we model sequences in language?